# 프로토콜 v1.1 — E2a: CruiseBench 시험 창에서 PI-BDP 채점 (탐색 분석)

CruiseBench(Cheng & Miao, arXiv 2607.19380) 표 6.1의 9개 서브셋 RMSE는 **논문 값을 그대로** 씁니다. 여기서는 PI-BDP만 같은 시험 창과 같은 상한으로 채점합니다. **학습이 없고 원시 HDF5도 필요 없습니다.**

- **창 재구성:** CruiseBench 공개 마스크(`common_altitude_cruise_mask.h5`, 커밋 `ef26d40`)의 시험 분할 순항 구간에서, 원 코드와 같이 사이클마다 길이 256·간격 10의 창 수를 셉니다. 창의 레이블은 그 사이클의 RUL에 서브셋별 η = 5% 상한을 씌운 값입니다(원 코드: 창 마지막 표본의 RUL, 한 사이클 안에서는 같음).
- **PI-BDP 예측 (v1.1 4.2절):** 사이클 t의 창에는 사이클 t − 1이 끝난 뒤의 예측(q500)에서 1을 뺀 값을 쓰고, 상한을 똑같이 적용합니다. 각 엔진의 첫 사이클 창은 뺍니다.
- **CruiseBench 수치의 성격:** 각 실행에서 시험 RMSE가 가장 좋은 epoch을 골라 보고한 값이라 낙관적입니다. 표 설명에 이 점을 적습니다.

**실행 전:** `ADDENDUM_COMMIT_URL`을 넣으세요. **결과:** 드라이브 `addendum_v11/e2a_*`. 런타임은 CPU면 됩니다.

In [ ]:
import os, sys, json, glob, time, hashlib, re
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
IN3 = f'{ROOT}/phase3'
OUT = f'{ROOT}/addendum_v11'; os.makedirs(OUT, exist_ok=True)
cfg = json.load(open(f'{ROOT}/phase1/config.json')); SUBSETS = sorted(cfg['k'])
K = ['subset', 'unit', 'cycle']

# 프로토콜 v1.1을 커밋한 주소 (실행 전에 커밋해야 합니다)
ADDENDUM_COMMIT_URL = ''
assert re.match(r'https://github\.com/.+/commit/[0-9a-f]{7,40}$', ADDENDUM_COMMIT_URL), \
    'v1.1 커밋 주소를 먼저 넣으세요. 추가본을 커밋하기 전에는 실행하지 않습니다.'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
# Phase 3 시험 파일이 고정 당시와 같은지 확인
REF = json.load(open(f'{IN3}/sha256_phase3.json'))
bad = [k for k, v in REF.items() if os.path.exists(f'{IN3}/{k}') and sha(f'{IN3}/{k}') != v]
assert not bad, f'Phase 3 결과 파일이 바뀌었습니다: {bad}'
print('ROOT', ROOT, '| 서브셋', SUBSETS, '| Phase 3 해시 확인', len(REF), '개')

In [ ]:
# === CruiseBench 저장소를 고정 커밋으로 받기 (마스크 파일만 씀) ===
import subprocess
CB = '/content/CruiseBench' if IN_COLAB else os.environ.get('CB_DIR', './CruiseBench')
CB_COMMIT = 'ef26d40'
if not os.path.exists(CB):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/NostalgiaJohn/CruiseBench', CB], check=True)
subprocess.run(['git', '-C', CB, 'fetch', '-q', '--depth', '50', 'origin'], check=False)
subprocess.run(['git', '-C', CB, 'checkout', '-q', CB_COMMIT], check=True)
head = subprocess.run(['git', '-C', CB, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head.startswith(CB_COMMIT), head
MASK = f'{CB}/dataset/cruise_mask/common_altitude_cruise_mask.h5'
print('CruiseBench', head, '| mask sha256', sha(MASK))

In [ ]:
# === 시험 창 수 재구성 (원 코드 data_load.py와 같은 규칙) ===
import h5py
W, STRIDE = 256, 10
CAPS = dict(zip(['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008'],
                [38, 34, 34, 43, 38, 41, 39, 34, 28]))           # 논문 표 4.1, eta = 5%
with h5py.File(MASK, 'r') as h: R = h['cycles'][()]
T = pd.DataFrame({'subset': [x.decode() for x in R['dataset']], 'split': [x.decode() for x in R['split']],
                  'unit': R['unit'], 'cycle': R['cycle'], 'detected': R['detected'].astype(bool),
                  'start': R['start_sample'], 'end': R['end_sample']})
# README 대조: 사이클 행 7,473, 순항 구간 검출 7,053
print('mask rows', len(T), '| detected', int(T.detected.sum()))
T = T[(T.split == 'test') & T.detected].copy()
T['len'] = T.end - T.start + 1
T['n_win'] = np.where(T.len >= W, (T.len - W) // STRIDE + 1, 0)
T = T[T.n_win > 0]
print(T.groupby('subset').agg(cycles=('cycle', 'size'), windows=('n_win', 'sum')).to_string())

In [ ]:
# === PI-BDP 고정 예측을 창에 놓기 (v1.1 4.2절) ===
V0 = pd.concat([pd.read_csv(f'{IN3}/test_PI-BDP_V0_{s}.csv.gz') for s in SUBSETS], ignore_index=True)[K + ['q500', 'true']]
eol = (V0.cycle + V0.true).groupby([V0.subset, V0.unit]).max()
# 마스크의 시험 엔진·사이클이 우리 시험 자료와 맞는지 확인
mx = T.groupby(['subset', 'unit']).cycle.max()
chk = pd.concat([mx.rename('mask_max_cycle'), eol.rename('eol')], axis=1)
print(chk.dropna().assign(ok=lambda d: d.mask_max_cycle <= d.eol).ok.all(), '| 엔진 수', len(chk.dropna()))
assert set(map(tuple, T[['subset', 'unit']].drop_duplicates().values)) <= set(eol.index), '마스크에 우리 시험 엔진이 아닌 것이 있음'
prev = V0.assign(cycle=V0.cycle + 1).rename(columns={'q500': 'q500_prev'})[K + ['q500_prev']]   # t-1 예측을 t로 옮김
S = T.merge(prev, on=K, how='left')
S['true'] = S.apply(lambda r: eol[(r.subset, r.unit)] - r.cycle, axis=1)
S['cap'] = S.subset.map(CAPS)
first = S.cycle == S.groupby(['subset', 'unit']).cycle.transform('min')
n_first = int(S[first].n_win.sum())
S = S[~first & S.q500_prev.notna()].copy()
S['label'] = np.minimum(S.true, S.cap)
S['pred'] = np.clip(S.q500_prev - 1, 0, S.cap)
print('채점 창', int(S.n_win.sum()), '| 뺀 첫 사이클 창', n_first)

In [ ]:
# === 창 가중 RMSE와 CruiseBench 표 6.1 ===
def wrmse(g): return float(np.sqrt(np.average((g.pred - g.label) ** 2, weights=g.n_win)))
pib = S.groupby('subset').apply(wrmse).rename('PI-BDP V0')
CB61 = pd.DataFrame({   # CruiseBench 논문 표 6.1 (eta5-W256-S10, 5회 평균 ± SD, 실행마다 최고 epoch)
    'LSTM':    [10.30, 6.17, 7.46, 13.97, 11.85, 13.69, 12.17, 10.76, 8.47],
    'GRU':     [6.69, 2.02, 2.33, 13.90, 4.44, 7.21, 8.63, 9.26, 5.68],
    'TCN':     [2.63, 2.52, 2.50, 6.98, 3.03, 3.05, 4.31, 6.92, 3.12],
    'TSMixer': [2.25, 2.41, 2.14, 4.72, 2.60, 2.61, 4.30, 7.32, 2.76]},
    index=['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008'])
TAB = CB61.join(pib)
TAB.loc['mean (9 subsets)'] = TAB.mean()
TAB.to_csv(f'{OUT}/e2a_table.csv'); S.to_csv(f'{OUT}/e2a_pibdp_windows.csv.gz', index=False)
pooled = float(np.sqrt(np.average((S.pred - S.label) ** 2, weights=S.n_win)))
print(TAB.round(2).to_string()); print('PI-BDP V0 창 가중 통합 RMSE %.3f' % pooled)

In [ ]:
# === 요약 저장 + 해시 ===
txt = '\n'.join([f"Addendum v1.1 E2a (exploratory) {time.strftime('%Y-%m-%d %H:%M:%S')}  addendum={ADDENDUM_COMMIT_URL}  CruiseBench={head}",
    'CruiseBench values: Table 6.1 of arXiv 2607.19380 as reported (best test epoch per run; optimistic).',
    'PI-BDP V0: frozen Phase 3 predictions; window at cycle t uses prediction after cycle t-1 minus 1; eta=5% caps; first-cycle windows excluded.',
    f'Scored windows {int(S.n_win.sum())}; excluded first-cycle windows {n_first}; pooled window-weighted RMSE {pooled:.4f}',
    '', TAB.round(3).to_string()])
open(f'{OUT}/e2a_summary.txt', 'w').write(txt)
H = {os.path.basename(p): sha(p) for p in sorted(glob.glob(f'{OUT}/e2a_*'))}
json.dump(H, open(f'{OUT}/sha256_e2a.json', 'w'), indent=1); print(txt)